# PKH Context Sniper — Colab launcher

Запуск ингеста и поиска по архиву чатов Google AI Studio.

**Перед запуском:** `Runtime → Change runtime type → T4 GPU`.

## 0. Bootstrap

Одна ячейка — монтирование, код, зависимости, конфиг, инициализация снайпера.
Запускай после каждого рестарта рантайма. Дальше — только нужные ячейки.

In [ ]:
# === BOOTSTRAP: запускай эту ячейку после каждого рестарта рантайма ===

# --- Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Clone / update repo ---
import os, sys
REPO_URL = 'https://github.com/LilSheva/PKH.git'
REPO_DIR = '/content/PKH'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# --- Dependencies (silent) ---
!pip install -q -r {REPO_DIR}/requirements.txt

# --- GPU check ---
import torch
print('CUDA:', torch.cuda.is_available(),
      f'| {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else '| NO GPU')

# --- Config ---
from pathlib import Path
import config

# >>> ПОДПРАВЬ ПОД СЕБЯ <<<
config.CHATS_DIR = Path('/content/drive/MyDrive/ai_studio_dumps')
config.DB_DIR = Path('/content/drive/MyDrive/pkh_chroma')
config.EMBEDDING_MODEL = 'bge-m3'       # bge-m3 | qwen3 | e5-instruct
config.THOUGHT_MODE = 'SMART'           # OFF | ON | SMART

# --- Init sniper ---
from sniper import ContextSniper
sniper = ContextSniper(embedding_model=config.EMBEDDING_MODEL)

print(f'\n✓ Ready | model={config.EMBEDDING_MODEL} | chats={config.CHATS_DIR}')
print(f'  db={config.DB_DIR} | thought_mode={config.THOUGHT_MODE}')

## 1. Ингест

Идемпотентно — повторные запуски обрабатывают только новые/изменённые файлы.
Первый раз модель скачается (~2 ГБ), дальше из кэша.

In [ ]:
stats = sniper.ingest(config.CHATS_DIR)
print(stats)
print(sniper.stats())

## 2. Поиск контекста

In [ ]:
query = 'как я настраивал ChromaDB на Colab'  # <<< свой запрос

res = sniper.search_context(query, top_k=5)
for i, (doc, meta, dist) in enumerate(zip(
    res['documents'][0], res['metadatas'][0], res['distances'][0]
)):
    print(f'--- #{i+1} | chat={meta["chat_id"]} | dist={dist:.3f} ---')
    print(doc[:500])
    print()

## 3. Супер-промпт

In [ ]:
prompt = sniper.generate_super_prompt(
    main_prompt='Напиши обёртку над ингестом, которая логирует прогресс по файлам.',
    meta_query='ингест файлов без расширения, прогресс-логирование',
)
print(prompt)

## 4. Тегирование чатов (Gemini)

Требует API-ключ. Добавь его в Colab Secrets (`🔑` слева) как `GEMINI_API_KEY`.

In [ ]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

def gemini_call(prompt: str) -> str:
    resp = genai.GenerativeModel('gemini-2.0-flash').generate_content(prompt)
    return resp.text

results = sniper.tag_chats(gemini_call, only_untagged=True)
for r in results:
    print(f'{r.chat_id}: {r.tags}')

## 5. Сравнение моделей

Каждая модель строит свою коллекцию — первый прогон по новой модели делает полный ингест.
**Внимание:** ×3 по времени и месту на Drive.

In [ ]:
MODELS = ['bge-m3', 'qwen3', 'e5-instruct']
QUERY = 'как я настраивал ChromaDB на Colab'

snipers = {}
for m in MODELS:
    print(f'=== ingest with {m} ===')
    s = ContextSniper(embedding_model=m)
    print(s.ingest(config.CHATS_DIR))
    snipers[m] = s

for m, s in snipers.items():
    print(f'\n========== {m} ==========')
    res = s.search_context(QUERY, top_k=3)
    for doc, meta, dist in zip(
        res['documents'][0], res['metadatas'][0], res['distances'][0]
    ):
        print(f'[{meta["chat_id"]}] dist={dist:.3f}  {doc[:200]}')